In [0]:
df = spark.read.format("csv").option("header",True).load("/Volumes/practice/default/data_sample/titanic_data.csv")

In [0]:
df.display()

#1 Calculate Running total

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
ex1 = (
    df.groupBy("embark_town").agg(
        sum(col("survived").cast('int')).alias("Survived")
    )
    .orderBy("Survived")
)
ex1.display()

In [0]:
## Window

window_spec = Window.orderBy("embark_town").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

run_total = ex1.withColumn(
    "Running_tot",
    sum("Survived").over(window_spec)
)
run_total.display()

In [0]:
df.createOrReplaceTempView("sql_table")

In [0]:
%sql
-- Same in SQL

with cte1 as (
select 
embark_town,
sum(cast(survived as int)) as Survived
from sql_table
group by 1
order by 1)
select
*,
sum(survived) over(order by embark_town) as Running_tot
from cte1

#02 Schema Inferencing

In [0]:
df.display(10)

In [0]:
from pyspark.sql.types import *
schema = StructType([
    StructField("survived",IntegerType()),
    StructField("pclass",IntegerType()),
    StructField("sex",StringType()),
    StructField("age",IntegerType()),
    StructField("sibsp",IntegerType()),
    StructField("parch",IntegerType()),
    StructField("fare",FloatType()),
    StructField("embarked",StringType()),
    StructField("class",StringType()),
    StructField("who",StringType()),
    StructField("adult_male",BooleanType()),
    StructField("deck",StringType()),
    StructField("embark_town",StringType()),
    StructField("alive",StringType()),
    StructField("alone",BooleanType()),
])

df = spark.read.format("csv").option("header",True).schema(schema).load("/Volumes/practice/default/data_sample/titanic_data.csv")

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").option("overwriteSchema",True).saveAsTable("practice.default.titanic_practice")

# 03 Implements SCD-Type 2

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("address",StringType()),
    StructField("city",StringType()),
    StructField("contact",StringType()),
    StructField("cust_type",StringType()),
    StructField("customer_id",IntegerType()),
    StructField("service_cd",StringType()),
    StructField("state",StringType())
])

In [0]:
## data load

df = spark.read.option('multiline',True).schema(schema).json('/Volumes/practice/default/data_sample/scd_examples.json')
df = df.withColumn(
    "contact",
    trim(col("contact")).cast('bigint')
)
df.display()

In [0]:
## adding effective date and expiry date

df_eff_dt = df.withColumn(
    "eff_dt",
    lit("2026-05-18").cast("date")
)
df_eff_dt = df_eff_dt.withColumn(
    "exp_dt",
    lit("9999-12-31").cast("date")
)
df_eff_dt.display()

In [0]:
## saving as delta table

df_eff_dt.write.option("overwriteSchema",True).mode("overwrite").saveAsTable("practice.default.cust_type2_table")

In [0]:
## creating scenerio where we are getting new and updated data
new_file = spark.read.option('multiline',True).schema(schema).json('/Volumes/practice/default/data_sample/case3.json')
hash_cols = ['address','city','contact','cust_type','customer_id','service_cd','state']

new_file = new_file.withColumn(
    "contact",
    trim(col("contact")).cast('bigint')
) \
.withColumn(
    "hashkey",
    sha2(
        concat_ws(
            "||",
            *[
                upper(trim(col(c).cast('string')))
                for c in hash_cols
            ]
        ),
        256
    )
)
new_file.display()

In [0]:
## finding new records
from delta.tables import DeltaTable
path = 'practice.default.cust_type2_table'
src_delta = DeltaTable.forName(spark,path)
src_tab = src_delta.toDF()
src_tab = (
    src_tab.withColumn(
        "hashkey",
        sha2(
            concat_ws(
                "||",
                *[
                    upper(trim(col(c).cast('string')))
                    for c in hash_cols
                ]
            ),
            256
        )
    )
)
src_tab.display()

In [0]:
changed_df = src_tab.alias("a").filter((col("exp_dt") > current_date()) & (col("eff_dt") < col("exp_dt"))).join(
    new_file.alias("b"),
    on=[(col("a.customer_id") == col("b.customer_id"))],
    how="inner"
) \
.filter(col("a.hashkey") != col("b.hashkey")).select("b.*")
changed_df.write.mode('overwrite').saveAsTable('practice.default.changed_df')
changed_df.display()

In [0]:
# Identify changed records

changed_df = src_tab.alias("a").filter((col("exp_dt") > current_date()) & (col("eff_dt") < col("exp_dt"))).join(
    new_file.alias("b"),
    on=[(col("a.customer_id") == col("b.customer_id"))],
    how="inner"
) \
.filter((col("a.hashkey") != col("b.hashkey")) & (col("a.cust_type") != col("b.cust_type")) & (col("a.service_cd") != col("b.service_cd"))).select("b.*")
changed_df.write.mode('overwrite').saveAsTable('practice.default.changed_df')
changed_df.display()

In [0]:
%sql
RESTORE TABLE practice.default.cust_type2_table TO VERSION AS OF 4;

In [0]:
%sql
select * from practice.default.cust_type2_table
order by customer_id

In [0]:
## Insert new + updated records
new_df = new_file.alias("a").join(
    src_tab.alias("b"),
    on="customer_id",
    how="left_anti"
)
new_df.display()

In [0]:
updated_records_df = changed_df
updated_records_df.display()

In [0]:
## Expire existing active records

src_delta.alias("a").merge(
    changed_df.alias("b"),
    "a.customer_id = b.customer_id and a.eff_dt < a.exp_dt and a.exp_dt > current_date"
).whenMatchedUpdate(
    set = {
       "exp_dt":current_date()
    }
).execute()

In [0]:
updated_records_df = spark.table('practice.default.changed_df')

final_df = new_df.unionByName(updated_records_df) \
    .withColumn(
        "eff_dt",
        current_date()
    ) \
    .withColumn(
        "exp_dt",
        lit("9999-12-31").cast("date")
    ) \
    .drop("hashkey")
final_df.display()

In [0]:
final_df.write.mode('append').saveAsTable(path)

In [0]:
%sql
select * from practice.default.cust_type2_table

In [0]:
%sql
select * from practice.default.cust_type2_table
where eff_dt < exp_dt and exp_dt > current_date

#04 Understanding delta table ans its concepts

### Difference Dataframe vs DeltaTable

| Feature                    | DataFrame | DeltaTable |
| -------------------------- | --------- | ---------- |
| In-memory abstraction      | Yes       | No         |
| Represents table storage   | No        | Yes        |
| Supports MERGE             | No        | Yes        |
| Supports DELETE/UPDATE     | No        | Yes        |
| ACID transactions          | No        | Yes        |
| Time travel                | No        | Yes        |
| Version history            | No        | Yes        |
| Mainly for transformations | Yes       | Partly     |
| Maintains metadata/logs    | No        | Yes        |


#05 Schema Evolution

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('csv').option('header',True).load('/Volumes/practice/default/data_sample/titanic_data.csv')
df.display()

In [0]:
df.columns

In [0]:
schema = StructType([
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), True),
    StructField("Parch", IntegerType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Embarked", StringType(), True)
])

df = spark.read.format('csv').option('header', True).schema(schema).load('/Volumes/practice/default/data_sample/titanic_data.csv')
df.display()

| Feature                        | mergeSchema | overwriteSchema |
| ------------------------------ | ----------- | --------------- |
| Adds new columns               | Yes         | Yes             |
| Removes old columns            | No          | Yes             |
| Preserves existing schema      | Yes         | No              |
| Replaces schema completely     | No          | Yes             |
| Typical mode                   | append      | overwrite       |
| Safer for production           | Usually yes | Riskier         |
| Existing data columns retained | Yes         | No              |


### Scenerio One

In [0]:
## split the dataset 

df_first500 = df.withColumn(
    "rnk",
    row_number().over(Window.orderBy('Age'))
) \
.filter(col('rnk') <= 500)
df_first500.write.mode('overwrite').option('mergeSchema',True).format('delta').saveAsTable('practice.default.titanic_delta_table')

In [0]:
%sql
select * from practice.default.titanic_delta_table

In [0]:
## scenerio, now new columns has been added to the new data, addition of new column should be done without breaking the pipeline

schema = StructType([
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), True),
    StructField("Parch", IntegerType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Embarked", StringType(), True),
    StructField("class",StringType(),True),
    StructField("who",StringType(),True),
    StructField('adult_male',BooleanType(),True)
])

new_data = spark.read.format('csv').option('header', True).schema(schema).load('/Volumes/practice/default/data_sample/titanic_data.csv')

new_data = new_data.withColumn(
    "rnk",
    row_number().over(Window.orderBy('Age'))
) \
.filter(col('rnk') > 500)
new_data.display()


In [0]:
new_data.write.mode('append').option('mergeSchema',True).format('delta').saveAsTable('practice.default.titanic_delta_table')

In [0]:
%sql
select * from practice.default.titanic_delta_table

### Scenerio Two (Backfill)

In [0]:
## split the dataset 
schema = StructType([
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("SibSp", IntegerType(), True),
    StructField("Parch", IntegerType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Embarked", StringType(), True)
])

df = spark.read.format('csv').option('header', True).schema(schema).load('/Volumes/practice/default/data_sample/titanic_data.csv')


df_first500 = df.withColumn(
    "rnk",
    row_number().over(Window.orderBy('Age'))
) \
.filter(col('rnk') <= 500)

df_first500.display()
df_first500.write.mode('overwrite').option('mergeSchema',True).option("overWriteSchema",True).format('delta').saveAsTable('practice.default.titanic_delta_table_case2')

In [0]:
%sql
select * from practice.default.titanic_delta_table_case2

In [0]:
## scenerio, now new columns has been added to the new data, addition of new column should be done without breaking the pipeline

schema = StructType([
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("SibSp", IntegerType(), True),
    StructField("Parch", IntegerType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Embarked", StringType(), True),
    StructField("class",StringType(),True),
    StructField("who",StringType(),True),
    StructField('adult_male',BooleanType(),True)
])

new_data = spark.read.format('csv').option('header', True).schema(schema).load('/Volumes/practice/default/data_sample/titanic_data.csv')

new_data = new_data.withColumn(
    "rnk",
    row_number().over(Window.orderBy('Age'))
)
new_data.display()


In [0]:
# Schema evolution for Delta merge operations
# Note: The spark.databricks.delta.schema.autoMerge.enabled config is not available on Serverless
# Instead, use SQL to set table property:
# ALTER TABLE practice.default.titanic_delta_table_case2 SET TBLPROPERTIES ('delta.autoMerge.enabled' = 'true');

# spark.sql("""
#     ALTER TABLE practice.default.titanic_delta_table_case2 
#     SET TBLPROPERTIES ('delta.autoMerge.enabled' = 'true')
# """)

In [0]:
## quick fix

empty_df = new_data.limit(0)
empty_df.write \
    .mode("append") \
        .option("mergeSchema", "true") \
            .format("delta") \
             .saveAsTable("practice.default.titanic_delta_table_case2")

In [0]:
%sql
select * from practice.default.titanic_delta_table_case2

In [0]:
new_data.display()

In [0]:
from delta.tables import DeltaTable
name = 'practice.default.titanic_delta_table_case2'
delta_tab = DeltaTable.forName(spark,name)

delta_tab.alias('dt').merge(
    new_data.alias('nd'),
    "dt.rnk = nd.rnk"
).whenNotMatchedInsert(
    values = {
        "Survived": "nd.Survived",
        "Pclass": "nd.Pclass",
        "Sex": "nd.Sex",
        "Age": "nd.Age",
        "SibSp": "nd.SibSp",
        "Parch": "nd.Parch",
        "Fare": "nd.Fare",
        "Embarked": "nd.Embarked",
        "class": "nd.class",
        "who": "nd.who",
        "adult_male": "nd.adult_male",
        "rnk":"nd.rnk"
    }
).whenMatchedUpdate(
    condition="""
    dt.class is NULL
    or
    dt.who is NULL
    or 
    dt.adult_male is NULL
    """,
    set = {
        "class": "nd.class",
        "who": "nd.who",
        "adult_male": "nd.adult_male"
    }
).execute()

In [0]:
%sql
select * from practice.default.titanic_delta_table_case2

In [0]:
%sql
describe history practice.default.titanic_delta_table_case2

In [0]:
%sql
restore table practice.default.titanic_delta_table_case2 to version as of 11

#06 Implement CDC

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
schema = StructType([
    StructField('Invoice_id', StringType(), True),
    StructField('Branch', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Customer_type', StringType(), True),
    StructField('Gender', StringType(), True),
    StructField('Product_line', StringType(), True),
    StructField('Unit_price', DoubleType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('Tax_percentage', DoubleType(), True),
    StructField('Total', DoubleType(), True),
    StructField('Date', StringType(), True),
    StructField('Time', StringType(), True),
    StructField('Payment', StringType(), True),
    StructField('cogs', DoubleType(), True),
    StructField('gross_margin_percentage', DoubleType(), True),
    StructField('gross_income', DoubleType(), True),
    StructField('Rating', DoubleType(), True)
])

In [0]:
## dataset EDA
df = spark.read.format('csv').option('header',True).schema(schema).load('/Volumes/practice/default/data_sample/supermarket_sales - Sheet1.csv')
df.display()

In [0]:
#create timestamp

df1 = (
    df
    .withColumn("purchase_time",
                to_timestamp(concat(col("Date"),lit(" "),col("time"),lit(":00")),"M/d/yyyy HH:mm:ss")
                )
    .drop("date","time")
    .limit(10)
)

df1.display()

In [0]:
df1.write.format("delta").mode("overwrite").saveAsTable("practice.default.supermarket_sales")

In [0]:
%sql
select * from practice.default.supermarket_sales

## CDC Pipeline

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

schema = StructType([
    StructField('Invoice_id', StringType(), True),
    StructField('Branch', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Customer_type', StringType(), True),
    StructField('Gender', StringType(), True),
    StructField('Product_line', StringType(), True),
    StructField('Unit_price', DoubleType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('Tax_percentage', DoubleType(), True),
    StructField('Total', DoubleType(), True),
    StructField('Payment', StringType(), True),
    StructField('cogs', DoubleType(), True),
    StructField('gross_margin_percentage', DoubleType(), True),
    StructField('gross_income', DoubleType(), True),
    StructField('Rating', DoubleType(), True),
    StructField('purchase_time', StringType(), True),
    StructField('op_type', StringType(), True)
])

In [0]:
cdc_df = spark.read.option('multiline',True).schema(schema).json("/Volumes/practice/default/data_sample/cdc_c3.json") \
    .withColumn(
        "purchase_time",
        try_to_timestamp(col("purchase_time"))
    )
cdc_df.display()

In [0]:
deltadf = DeltaTable.forName(spark,"practice.default.supermarket_sales")
deltadf.alias("sm").merge(
    cdc_df.alias("cdc"),
    condition="sm.invoice_id = cdc.invoice_id"
) \
    .whenMatchedDelete(
        condition="cdc.op_type = 'D'",
    ) \
    .whenMatchedUpdate(
        condition="cdc.op_type = 'U'",
        set = {
            "Invoice_id": col("cdc.invoice_id"),
            "Branch": col("cdc.Branch"),
            "City": col("cdc.City"),
            "Customer_type": col("cdc.Customer_type"),
            "Gender": col("cdc.Gender"),
            "Product_line": col("cdc.Product_line"),
            "Unit_price": col("cdc.Unit_price"),
            "Quantity": col("cdc.Quantity"),
            "Tax_percentage": col("cdc.Tax_percentage"),
            "Total": col("cdc.Total"),
            "Payment": col("cdc.Payment"),
            "cogs": col("cdc.cogs"),
            "gross_margin_percentage": col("cdc.gross_margin_percentage"),
            "gross_income": col("cdc.gross_income"),
            "Rating": col("cdc.Rating"),
            "purchase_time": col("cdc.purchase_time")
        }
    ) \
    .whenNotMatchedInsert(
        values = {
            "Invoice_id" : col("cdc.invoice_id"),
            "Branch": col("cdc.Branch"),
            "City": col("cdc.City"),
            "Customer_type": col("cdc.Customer_type"),
            "Gender": col("cdc.Gender"),
            "Product_line": col("cdc.Product_line"),
            "Unit_price": col("cdc.Unit_price"),
            "Quantity": col("cdc.Quantity"),
            "Tax_percentage": col("cdc.Tax_percentage"),
            "Total": col("cdc.Total"),
            "Payment": col("cdc.Payment"),
            "cogs": col("cdc.cogs"),
            "gross_margin_percentage": col("cdc.gross_margin_percentage"),
            "gross_income": col("cdc.gross_income"),
            "Rating": col("cdc.Rating"),
            "purchase_time": col("cdc.purchase_time")
        }
    ) \
    .execute()

In [0]:
%sql
select * from practice.default.supermarket_sales

In [0]:
%sql
describe history practice.default.supermarket_sales

In [0]:
%sql
restore table practice.default.supermarket_sales to version as of 0